<a href="https://colab.research.google.com/github/kevinketerlondon-tech/Data-Analyst-portfolio/blob/main/API_SPY.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

!pip install alpaca-py --quiet
!pip install yfinance --quiet
!pip install fredapi --quiet

import pandas as pd
import numpy as np
import yfinance as yf
import joblib
import torch
import torch.nn as nn
import warnings
import os
from datetime import datetime, timedelta
from fredapi import Fred

warnings.filterwarnings('ignore')

print("✅ All imports ready")
print(f"   PyTorch version: {torch.__version__}")
print(f"   Run time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

Mounted at /content/drive
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 122.5/122.5 kB 6.2 MB/s eta 0:00:00
✅ All imports ready
   PyTorch version: 2.11.0+cpu
   Run time: 2026-06-07 22:03:42


In [2]:
# ══════════════════════════════════════════════════════════
# YOUR KEYS AND SETTINGS — FILL THESE IN
# ══════════════════════════════════════════════════════════
ALPACA_API_KEY    = 'PKPKTSVVJWKAUDDBY554P7DH4P'
ALPACA_SECRET_KEY = 'D4xFfif5zxW6jsFNcXSA5piyZ1E9BmQR1QrHDCwN3ZsG'
FRED_KEY          = 'b6f7e7e3115895b116562f3d7683c5fd'
PAPER_TRADING     = True

TICKER              = 'SPY'
RISK_PER_TRADE_PCT  = 0.02
CONFIDENCE_THRESHOLD= 0.45
LOOKBACK_DAYS       = '3y'
NORM_WINDOW         = 252

base_data   = '/content/drive/MyDrive/Dissertation/data/'
base_models = '/content/drive/MyDrive/Dissertation/models/'

print("✅ Settings loaded")
print(f"   Mode: {'PAPER TRADING' if PAPER_TRADING else '⚠️  LIVE'}")

# ── Verify model files exist before going further ─────────
needed_models = [
    'xgboost_model.pkl',
    'lstm_model.pt',
    'cnn_model.pt',
    'meta_learner.pkl',
    'meta_scaler.pkl'
]
all_found = True
print("\nChecking model files:")
for f in needed_models:
    path = base_models + f
    if os.path.exists(path):
        size = os.path.getsize(path) / 1024
        print(f"  ✅ {f:<25s}  {size:.1f} KB")
    else:
        print(f"  ❌ {f:<25s}  NOT FOUND")
        all_found = False

if not all_found:
    raise FileNotFoundError(
        "Missing model files above — "
        "re-run the Phase 3 notebooks that save them"
    )
print("\n✅ All model files present")

✅ Settings loaded
   Mode: PAPER TRADING

Checking model files:
  ✅ xgboost_model.pkl          124.2 KB
  ✅ lstm_model.pt              259.6 KB
  ✅ cnn_model.pt               451.1 KB
  ✅ meta_learner.pkl           1.1 KB
  ✅ meta_scaler.pkl            1.2 KB

✅ All model files present


In [3]:
from alpaca.trading.client   import TradingClient
from alpaca.trading.requests import (MarketOrderRequest,
                                      GetOrdersRequest)
from alpaca.trading.enums    import (OrderSide,
                                      TimeInForce,
                                      QueryOrderStatus)

trading_client = TradingClient(
    ALPACA_API_KEY,
    ALPACA_SECRET_KEY,
    paper=PAPER_TRADING
)

account = trading_client.get_account()

print("✅ Connected to Alpaca")
print(f"   Status:         {account.status}")
print(f"   Portfolio:      ${float(account.portfolio_value):,.2f}")
print(f"   Cash:           ${float(account.cash):,.2f}")
print(f"   Buying power:   ${float(account.buying_power):,.2f}")

# Check current SPY position
try:
    pos              = trading_client.get_open_position(TICKER)
    current_position = float(pos.qty)
    entry_price      = float(pos.avg_entry_price)
    unreal_pl        = float(pos.unrealized_pl)
    print(f"\n   {TICKER} position: {current_position} shares")
    print(f"   Entry price:    ${entry_price:.2f}")
    print(f"   Unrealised P&L: ${unreal_pl:.2f}")
except Exception:
    current_position = 0.0
    print(f"\n   No open {TICKER} position")

✅ Connected to Alpaca
   Status:         AccountStatus.ACTIVE
   Portfolio:      $94,564.47
   Cash:           $-99,411.18
   Buying power:   $145,487.10

   SPY position: 263.0 shares
   Entry price:    $758.22
   Unrealised P&L: $-5435.52


In [4]:
# ══════════════════════════════════════════════════════════
# THIS CELL BUILDS EVERY SINGLE FEATURE
# Matches Phase 2 exactly — same indicators, same macro,
# same stationarity fixes, same normalisation
# ══════════════════════════════════════════════════════════

fred_client = Fred(api_key=FRED_KEY)
start_str   = (datetime.today() - timedelta(days=1200)
               ).strftime('%Y-%m-%d')

print("Building complete feature set...")
print("=" * 55)

# ── BLOCK 1: SPY price data ────────────────────────────────
print("\n[1/6] SPY price data...")
raw           = yf.download(
    TICKER, period=LOOKBACK_DAYS,
    auto_adjust=True, progress=False,
    multi_level_index=False
)
df            = raw[['Open','High','Low','Close','Volume']].copy()
df.columns    = ['open','high','low','close','volume']
df.index      = pd.to_datetime(df.index)
df.index.name = 'date'
df.dropna(inplace=True)

# Save raw unnormalised values for position sizing later
raw_close = df['close'].copy()
raw_atr_series = None

close  = df['close']
high   = df['high']
low    = df['low']
volume = df['volume']
print(f"   ✅ {len(df)} days")

# ── BLOCK 2: All technical indicators ─────────────────────
print("\n[2/6] Technical indicators...")

def calc_rsi(series, period=14):
    delta    = series.diff()
    gain     = delta.clip(lower=0)
    loss     = -delta.clip(upper=0)
    avg_gain = gain.ewm(alpha=1/period,
                        min_periods=period).mean()
    avg_loss = loss.ewm(alpha=1/period,
                        min_periods=period).mean()
    rs       = avg_gain / avg_loss.replace(0, np.nan)
    return 100 - (100 / (1 + rs))

def calc_ema(series, span):
    return series.ewm(
        span=span, adjust=False, min_periods=span).mean()

df['rsi_14']      = calc_rsi(close, 14)
ema12             = calc_ema(close, 12)
ema26             = calc_ema(close, 26)
df['macd_line']   = ema12 - ema26
df['macd_signal'] = calc_ema(df['macd_line'], 9)
df['macd_hist']   = df['macd_line'] - df['macd_signal']

bb_mid            = close.rolling(20).mean()
bb_std            = close.rolling(20).std(ddof=0)
df['bb_upper']    = bb_mid + 2 * bb_std
df['bb_middle']   = bb_mid
df['bb_lower']    = bb_mid - 2 * bb_std
df['bb_width']    = ((df['bb_upper'] - df['bb_lower'])
                     / df['bb_middle'])
df['bb_pct']      = ((close - df['bb_lower'])
                     / (df['bb_upper']
                        - df['bb_lower']).replace(0, np.nan))

prev_close        = close.shift(1)
tr                = pd.concat([
    high - low,
    (high - prev_close).abs(),
    (low  - prev_close).abs()
], axis=1).max(axis=1)
df['atr_14']      = tr.ewm(alpha=1/14, min_periods=14).mean()

# Save raw ATR for position sizing
raw_atr_series    = tr.ewm(alpha=1/14, min_periods=14).mean()

df['obv']         = (np.sign(close.diff()
                             ).fillna(0) * volume).cumsum()
df['returns']     = close.pct_change()
df['rolling_vol_5']  = df['returns'].rolling(5).std()
df['rolling_vol_10'] = df['returns'].rolling(10).std()
df['rolling_vol_20'] = df['returns'].rolling(20).std()
df['rolling_vol_60'] = df['returns'].rolling(60).std()

df['ema_9']           = calc_ema(close, 9)
df['ema_21']          = calc_ema(close, 21)
df['ema_50']          = calc_ema(close, 50)
df['ema_200']         = calc_ema(close, 200)
df['price_vs_ema50']  = ((close - df['ema_50'])
                          / df['ema_50'])
df['price_vs_ema200'] = ((close - df['ema_200'])
                          / df['ema_200'])
df['ema50_vs_ema200'] = ((df['ema_50'] - df['ema_200'])
                          / df['ema_200'])

low14             = low.rolling(14).min()
high14            = high.rolling(14).max()
raw_k             = (100 * (close - low14)
                     / (high14 - low14).replace(0, np.nan))
df['stoch_k']     = raw_k.rolling(3).mean()
df['stoch_d']     = df['stoch_k'].rolling(3).mean()
df['williams_r']  = (-100 * (high14 - close)
                     / (high14 - low14).replace(0, np.nan))

tp                = (high + low + close) / 3
tp_ma             = tp.rolling(20).mean()
tp_md             = tp.rolling(20).apply(
    lambda x: np.mean(np.abs(x - x.mean())), raw=True)
df['cci_20']      = ((tp - tp_ma)
                     / (0.015 * tp_md.replace(0, np.nan)))

df['high_low_range'] = (high - low) / close
df['close_vs_open']  = (close - df['open']) / df['open']
df['volume_ma_20']   = volume.rolling(20).mean()
df['volume_ratio']   = (volume
                        / df['volume_ma_20'].replace(0, np.nan))
print(f"   ✅ Done")

# ── BLOCK 3: Macro data ────────────────────────────────────
print("\n[3/6] Macro data (FRED + Yahoo)...")

def safe_fred(series_id, name):
    try:
        s      = fred_client.get_series(
            series_id, observation_start=start_str)
        s.name = name
        return s
    except Exception as e:
        print(f"   ⚠️  {name}: {e} — zeros used")
        return pd.Series(dtype=float, name=name)

def safe_yahoo(ticker, name):
    try:
        r      = yf.download(
            ticker, period=LOOKBACK_DAYS,
            auto_adjust=True, progress=False,
            multi_level_index=False)
        s      = r['Close'].copy()
        s.name = name
        s.index= pd.to_datetime(s.index)
        return s
    except Exception as e:
        print(f"   ⚠️  {name}: {e} — zeros used")
        return pd.Series(dtype=float, name=name)

vix    = safe_fred('VIXCLS',   'vix')
t10y2y = safe_fred('T10Y2Y',   'yield_curve_10y2y')
dff    = safe_fred('DFF',      'fed_funds_rate')
dgs10  = safe_fred('DGS10',    'treasury_10y')
dgs2   = safe_fred('DGS2',     'treasury_2y')
unrate = safe_fred('UNRATE',   'unemployment')
lqd_s  = safe_yahoo('LQD',      'lqd_price')
hyg_s  = safe_yahoo('HYG',      'hyg_price')
dxy_s  = safe_yahoo('DX-Y.NYB', 'dxy')
gold_s = safe_yahoo('GLD',      'gold')

macro_map = {
    'vix'              : vix,
    'yield_curve_10y2y': t10y2y,
    'fed_funds_rate'   : dff,
    'treasury_10y'     : dgs10,
    'treasury_2y'      : dgs2,
    'unemployment'     : unrate,
    'lqd_price'        : lqd_s,
    'hyg_price'        : hyg_s,
    'dxy'              : dxy_s,
    'gold'             : gold_s,
}

for col, series in macro_map.items():
    if len(series) == 0:
        df[col] = 0.0
    else:
        df[col] = (series
                   .reindex(df.index)
                   .ffill(limit=10)
                   .bfill(limit=10))

df.fillna(0, inplace=True)
print(f"   ✅ Done")

# ── BLOCK 4: Derived macro features ───────────────────────
print("\n[4/6] Derived macro features...")

df['spread_10y_2y']  = df['treasury_10y'] - df['treasury_2y']
df['hyg_return_5d']  = df['hyg_price'].pct_change(5)
df['lqd_return_5d']  = df['lqd_price'].pct_change(5)
df['hyg_vs_lqd']     = (df['hyg_price']
                         / df['lqd_price'].replace(0, np.nan))
df['vix_change_1d']  = df['vix'].pct_change(1)
df['vix_change_5d']  = df['vix'].pct_change(5)
df['yield_chg_5d']   = df['yield_curve_10y2y'].diff(5)
df['dxy_change_5d']  = df['dxy'].pct_change(5)
df['gold_change_5d'] = df['gold'].pct_change(5)
df['sentiment_score']  = 0.0
df['sentiment_volume'] = 0.0
df['sentiment_std']    = 0.0
df.fillna(0, inplace=True)
print(f"   ✅ Done")

# ── BLOCK 5: Stationarity fixes ────────────────────────────
print("\n[5/6] Stationarity fixes...")

pct_cols = [
    'open','high','low','close',
    'lqd_price','hyg_price','gold','dxy',
    'treasury_10y','treasury_2y',
    'ema_9','ema_21','ema_50','ema_200',
    'bb_upper','bb_middle','bb_lower',
    'volume','volume_ma_20','obv'
]
diff_cols = [
    'yield_curve_10y2y','fed_funds_rate',
    'spread_10y_2y','hyg_vs_lqd'
]

for col in pct_cols:
    if col in df.columns:
        df[col + '_pct'] = df[col].pct_change()

for col in diff_cols:
    if col in df.columns:
        df[col + '_diff'] = df[col].diff()

df.fillna(0, inplace=True)
print(f"   ✅ Done")

# ── BLOCK 6: Rolling z-score normalisation ─────────────────
print("\n[6/6] Normalising features...")

skip_norm = set(pct_cols)

for col in df.columns:
    if col in skip_norm:
        continue
    if df[col].dtype not in [np.float64, np.float32,
                              float, np.int64]:
        continue
    roll_mean    = df[col].rolling(NORM_WINDOW).mean()
    roll_std     = (df[col].rolling(NORM_WINDOW)
                    .std().replace(0, np.nan))
    df[col]      = (df[col] - roll_mean) / roll_std

df.fillna(0, inplace=True)
print(f"   ✅ Done")

# ── Extract today values ───────────────────────────────────
today_date  = df.index[-1].date()
today_price = float(raw_close.iloc[-1])
today_atr   = float(raw_atr_series.iloc[-1])

# ══════════════════════════════════════════════════════════
# LOAD ALL MODELS
# ══════════════════════════════════════════════════════════
print("\nLoading models...")

# XGBoost
model_xgb    = joblib.load(base_models + 'xgboost_model.pkl')
feature_cols = list(model_xgb.feature_names_in_)

# Check for missing features and fill with 0
missing = [f for f in feature_cols if f not in df.columns]
if missing:
    print(f"   ⚠️  {len(missing)} features missing — filling with 0:")
    for m in missing:
        print(f"      - {m}")
        df[m] = 0.0
else:
    print(f"   ✅ All {len(feature_cols)} features present")

# Today's row for XGBoost
today_row = df.iloc[[-1]][feature_cols].fillna(0)

# 30-day sequence for LSTM and CNN
X_recent   = (df.tail(30)[feature_cols]
              .fillna(0).values.astype(np.float32))
n_features = X_recent.shape[1]
seq_lstm   = torch.tensor(
    X_recent, dtype=torch.float32).unsqueeze(0)
seq_cnn    = torch.tensor(
    X_recent, dtype=torch.float32).unsqueeze(0).permute(0, 2, 1)

# Meta-learner and scaler
meta_lr = joblib.load(base_models + 'meta_learner.pkl')
scaler  = joblib.load(base_models + 'meta_scaler.pkl')
print("   ✅ XGBoost, meta-learner, scaler loaded")

# ── LSTM architecture (must match Phase 3b exactly) ────────
class LSTMClassifier(nn.Module):
    def __init__(self, input_size, hidden_size,
                 num_layers, num_classes, dropout):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size  = input_size,
            hidden_size = hidden_size,
            num_layers  = num_layers,
            batch_first = True,
            dropout     = dropout if num_layers > 1 else 0.0
        )
        self.dropout = nn.Dropout(dropout)
        self.fc      = nn.Linear(hidden_size, num_classes)
    def forward(self, x):
        out, _ = self.lstm(x)
        return self.fc(self.dropout(out[:, -1, :]))

# ── CNN architecture (must match Phase 3c exactly) ─────────
class CNNClassifier(nn.Module):
    def __init__(self, n_features, n_classes):
        super().__init__()
        self.conv1 = nn.Sequential(
            nn.Conv1d(n_features, 64, 3, padding=1),
            nn.BatchNorm1d(64), nn.ReLU(), nn.Dropout(0.2))
        self.conv2 = nn.Sequential(
            nn.Conv1d(64, 128, 5, padding=2),
            nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(0.2))
        self.conv3 = nn.Sequential(
            nn.Conv1d(128, 64, 7, padding=3),
            nn.BatchNorm1d(64), nn.ReLU(), nn.Dropout(0.2))
        self.gap   = nn.AdaptiveAvgPool1d(1)
        self.fc    = nn.Sequential(
            nn.Linear(64, 32), nn.ReLU(),
            nn.Dropout(0.3), nn.Linear(32, n_classes))
    def forward(self, x):
        x = self.conv3(self.conv2(self.conv1(x)))
        return self.fc(self.gap(x).squeeze(-1))

# Load LSTM
# weights_only=False required for PyTorch 2.6 on Colab
model_lstm = LSTMClassifier(n_features, 64, 2, 3, 0.3)
model_lstm.load_state_dict(
    torch.load(
        base_models + 'lstm_model.pt',
        map_location='cpu',
        weights_only=False      # ← fixes PyTorch 2.6 crash
    )
)
model_lstm.eval()
print("   ✅ LSTM loaded")

# Load CNN
model_cnn = CNNClassifier(n_features, 3)
model_cnn.load_state_dict(
    torch.load(
        base_models + 'cnn_model.pt',
        map_location='cpu',
        weights_only=False      # ← fixes PyTorch 2.6 crash
    )
)
model_cnn.eval()
print("   ✅ CNN loaded")

# ══════════════════════════════════════════════════════════
# RUN INFERENCE — ALL THREE MODELS
# ══════════════════════════════════════════════════════════
print("\nRunning inference...")

# XGBoost
xgb_proba = model_xgb.predict_proba(today_row)[0]

# LSTM
with torch.no_grad():
    lstm_proba = torch.softmax(
        model_lstm(seq_lstm), dim=1).numpy()[0]

# CNN
with torch.no_grad():
    cnn_proba = torch.softmax(
        model_cnn(seq_cnn), dim=1).numpy()[0]

# Meta-learner combines all three
stacked = np.array([[
    xgb_proba[0],  xgb_proba[1],  xgb_proba[2],
    lstm_proba[0], lstm_proba[1], lstm_proba[2],
    cnn_proba[0],  cnn_proba[1],  cnn_proba[2]
]])

final_proba  = meta_lr.predict_proba(
    scaler.transform(stacked))[0]
confidence   = float(final_proba.max())
pred_enc     = int(np.argmax(final_proba))
label_map    = {0: 'SELL', 1: 'OUT', 2: 'BUY'}
raw_signal   = label_map[pred_enc]

# Apply confidence threshold
if confidence < CONFIDENCE_THRESHOLD:
    final_signal = 'OUT'
    override     = True
else:
    final_signal = raw_signal
    override     = False

# Individual model votes
votes = {
    'XGBoost': label_map[int(np.argmax(xgb_proba))],
    'LSTM'   : label_map[int(np.argmax(lstm_proba))],
    'CNN'    : label_map[int(np.argmax(cnn_proba))],
}

# ══════════════════════════════════════════════════════════
# SUMMARY
# ══════════════════════════════════════════════════════════
print(f"\n{'='*55}")
print(f"  FEATURE AND INFERENCE COMPLETE")
print(f"  Date:          {today_date}")
print(f"  SPY close:     ${today_price:.2f}")
print(f"  ATR:           ${today_atr:.2f}")
print(f"  Features:      {len(feature_cols)}")
print(f"  XGBoost:       {votes['XGBoost']}  "
      f"({xgb_proba.max()*100:.1f}%)")
print(f"  LSTM:          {votes['LSTM']}  "
      f"({lstm_proba.max()*100:.1f}%)")
print(f"  CNN:           {votes['CNN']}  "
      f"({cnn_proba.max()*100:.1f}%)")
print(f"  Final signal:  {final_signal}  "
      f"({confidence*100:.1f}%)")
print(f"  Overridden:    {override}")
print(f"{'='*55}")
print("✅ Ready for Cell 5 — position sizing and dashboard")

Building complete feature set...

[1/6] SPY price data...
   ✅ 753 days

[2/6] Technical indicators...
   ✅ Done

[3/6] Macro data (FRED + Yahoo)...
   ✅ Done

[4/6] Derived macro features...
   ✅ Done

[5/6] Stationarity fixes...
   ✅ Done

[6/6] Normalising features...
   ✅ Done

Loading models...
   ✅ All 60 features present
   ✅ XGBoost, meta-learner, scaler loaded
   ✅ LSTM loaded
   ✅ CNN loaded

Running inference...

  FEATURE AND INFERENCE COMPLETE
  Date:          2026-06-05
  SPY close:     $737.55
  ATR:           $7.61
  Features:      60
  XGBoost:       BUY  (42.8%)
  LSTM:          BUY  (40.5%)
  CNN:           SELL  (38.3%)
  Final signal:  BUY  (56.4%)
  Overridden:    False
✅ Ready for Cell 5 — position sizing and dashboard


In [5]:
portfolio_val  = float(account.portfolio_value)
cash_available = float(account.cash)
risk_amount    = portfolio_val * RISK_PER_TRADE_PCT
stop_distance  = 2.0 * today_atr
shares         = max(1, round(risk_amount / stop_distance))
trade_value    = shares * today_price

stop_loss_price   = round(today_price - stop_distance, 2)
take_profit_price = round(today_price + stop_distance, 2)

# Cap at available cash
if trade_value > cash_available and final_signal == 'BUY':
    shares      = max(1, int(cash_available / today_price))
    trade_value = shares * today_price
    print(f"⚠️  Capped to {shares} shares due to cash limit")

print(f"Position sizing:")
print(f"  Portfolio:    ${portfolio_val:,.2f}")
print(f"  Risk amount:  ${risk_amount:,.2f}")
print(f"  ATR:          ${today_atr:.2f}")
print(f"  Stop dist:    ${stop_distance:.2f}")
print(f"  Shares:       {shares}")
print(f"  Trade value:  ${trade_value:,.2f}")
print(f"  Stop loss:    ${stop_loss_price:.2f}")
print(f"  Take profit:  ${take_profit_price:.2f}")

⚠️  Capped to 1 shares due to cash limit
Position sizing:
  Portfolio:    $94,564.47
  Risk amount:  $1,891.29
  ATR:          $7.61
  Stop dist:    $15.22
  Shares:       1
  Trade value:  $737.55
  Stop loss:    $722.33
  Take profit:  $752.77


In [6]:
signal_emoji = {'BUY': '🟢', 'SELL': '🔴', 'OUT': '⚪'}

print("╔══════════════════════════════════════════════════╗")
print("║         TRADING DECISION DASHBOARD              ║")
print("╠══════════════════════════════════════════════════╣")
print(f"║  Date:   {today_date}                              ║")
print(f"║  SPY:    ${today_price:<8.2f}                         ║")
print(f"║  ATR:    ${today_atr:<8.2f}                         ║")
print("╠══════════════════════════════════════════════════╣")
print("║  MODEL VOTES                                    ║")
for m, v in votes.items():
    conf = {'XGBoost':xgb_proba,
            'LSTM':lstm_proba,
            'CNN':cnn_proba}[m]
    print(f"║  {m:<10s} {signal_emoji[v]} {v:<4s}  "
          f"{conf.max()*100:.1f}% confident             ║")
print("╠══════════════════════════════════════════════════╣")
print("║  ENSEMBLE PROBABILITIES                         ║")
print(f"║    SELL: {final_proba[0]*100:5.1f}%                          ║")
print(f"║    OUT:  {final_proba[1]*100:5.1f}%                          ║")
print(f"║    BUY:  {final_proba[2]*100:5.1f}%                          ║")
print("╠══════════════════════════════════════════════════╣")
if override:
    print(f"║  ⚠️  OVERRIDDEN TO OUT                          ║")
    print(f"║     confidence {confidence*100:.1f}% < "
          f"threshold {CONFIDENCE_THRESHOLD*100:.0f}%          ║")
else:
    print(f"║  SIGNAL: {signal_emoji[final_signal]} "
          f"{final_signal:<4s}  ({confidence*100:.1f}% confident)   ║")
print("╠══════════════════════════════════════════════════╣")
if final_signal == 'BUY':
    print(f"║  TRADE:  BUY {shares} shares @ ~${today_price:.2f}       ║")
    print(f"║  Cost:   ${trade_value:,.2f}                       ║")
    print(f"║  Stop:   ${stop_loss_price:.2f}                         ║")
    print(f"║  Target: ${take_profit_price:.2f}                         ║")
elif final_signal == 'SELL' and current_position > 0:
    print(f"║  TRADE:  CLOSE {current_position} shares             ║")
elif final_signal == 'SELL' and current_position == 0:
    print(f"║  SELL signal — no position to close             ║")
    print(f"║  Action: HOLD CASH                              ║")
else:
    print(f"║  Action: DO NOTHING                             ║")
print("╠══════════════════════════════════════════════════╣")
print(f"║  Cash:      ${cash_available:>10,.2f}                  ║")
print(f"║  Portfolio: ${portfolio_val:>10,.2f}                  ║")
print(f"║  {TICKER} held: {current_position:>6.1f} shares                   ║")
print("╚══════════════════════════════════════════════════╝")
print()
print("  Change YOUR_DECISION in Cell 7 to 'yes' or 'no'")

╔══════════════════════════════════════════════════╗
║         TRADING DECISION DASHBOARD              ║
╠══════════════════════════════════════════════════╣
║  Date:   2026-06-05                              ║
║  SPY:    $737.55                           ║
║  ATR:    $7.61                             ║
╠══════════════════════════════════════════════════╣
║  MODEL VOTES                                    ║
║  XGBoost    🟢 BUY   42.8% confident             ║
║  LSTM       🟢 BUY   40.5% confident             ║
║  CNN        🔴 SELL  38.3% confident             ║
╠══════════════════════════════════════════════════╣
║  ENSEMBLE PROBABILITIES                         ║
║    SELL:  26.2%                          ║
║    OUT:   17.4%                          ║
║    BUY:   56.4%                          ║
╠══════════════════════════════════════════════════╣
║  SIGNAL: 🟢 BUY   (56.4% confident)   ║
╠══════════════════════════════════════════════════╣
║  TRADE:  BUY 1 shares @ ~$737.55       ║
║  C

In [7]:
# ══════════════════════════════════════════════════════════
# READ THE DASHBOARD ABOVE THEN SET YOUR DECISION
# 'yes'  → place the trade
# 'no'   → skip
# 'info' → print extra market context
# ══════════════════════════════════════════════════════════

YOUR_DECISION = 'YES'   # ← CHANGE THIS EACH TIME

decision    = YOUR_DECISION.strip().lower()
order_placed = False

if decision == 'info':
    print("Extra market context:")
    print(f"  RSI:             {df['rsi_14'].iloc[-1]:.2f}")
    print(f"  MACD histogram:  {df['macd_hist'].iloc[-1]:.5f}")
    print(f"  BB%:             {df['bb_pct'].iloc[-1]:.3f}")
    print(f"  EMA50 vs EMA200: {df['ema50_vs_ema200'].iloc[-1]:.5f}")
    print(f"  Volume ratio:    {df['volume_ratio'].iloc[-1]:.2f}x")
    print(f"  20d volatility:  {df['rolling_vol_20'].iloc[-1]*100:.3f}%")
    print("\nChange YOUR_DECISION to 'yes' or 'no' and rerun")

elif decision == 'yes':
    if final_signal == 'OUT':
        print("⚪ Signal is OUT — no trade placed")

    elif final_signal == 'BUY':
        if shares <= 0:
            print("❌ 0 shares — cannot place order")
        else:
            print(f"Placing BUY: {shares} x {TICKER}...")
            try:
                order = trading_client.submit_order(
                    MarketOrderRequest(
                        symbol        = TICKER,
                        qty           = shares,
                        side          = OrderSide.BUY,
                        time_in_force = TimeInForce.DAY
                    )
                )
                order_placed = True
                print(f"✅ BUY ORDER PLACED")
                print(f"   Order ID: {order.id}")
                print(f"   Qty:      {order.qty}")
                print(f"   Status:   {order.status}")
            except Exception as e:
                print(f"❌ Order failed: {e}")

    elif final_signal == 'SELL':
        if current_position > 0:
            print(f"Closing {TICKER} position...")
            try:
                trading_client.close_position(TICKER)
                order_placed = True
                print(f"✅ POSITION CLOSED")
            except Exception as e:
                print(f"❌ Close failed: {e}")
        else:
            print("ℹ️  SELL signal but no position — holding cash")

elif decision == 'no':
    print("⏭️  Skipped by user — no order placed")

else:
    print(f"❌ Unknown input '{YOUR_DECISION}' — use yes/no/info")

Placing BUY: 1 x SPY...
✅ BUY ORDER PLACED
   Order ID: 7c7f08f5-eee1-40f6-8155-256462c7298f
   Qty:      1
   Status:   OrderStatus.ACCEPTED


In [8]:
import time

# Log this run regardless of decision
log_entry = {
    'datetime'      : datetime.now().strftime('%Y-%m-%d %H:%M'),
    'date'          : str(today_date),
    'spy_close'     : today_price,
    'atr'           : round(today_atr, 4),
    'xgb_vote'      : votes['XGBoost'],
    'lstm_vote'     : votes['LSTM'],
    'cnn_vote'      : votes['CNN'],
    'raw_signal'    : raw_signal,
    'confidence'    : round(confidence, 4),
    'final_signal'  : final_signal,
    'overridden'    : override,
    'p_sell'        : round(float(final_proba[0]), 4),
    'p_out'         : round(float(final_proba[1]), 4),
    'p_buy'         : round(float(final_proba[2]), 4),
    'shares'        : shares if final_signal == 'BUY' else 0,
    'trade_value'   : (trade_value
                       if final_signal == 'BUY' else 0),
    'stop_loss'     : stop_loss_price,
    'take_profit'   : take_profit_price,
    'decision'      : decision,
    'order_placed'  : order_placed,
    'portfolio'     : portfolio_val,
    'cash'          : cash_available,
}

log_path = base_data + 'paper_trading_log.csv'
if os.path.exists(log_path):
    log = pd.concat(
        [pd.read_csv(log_path),
         pd.DataFrame([log_entry])],
        ignore_index=True
    )
else:
    log = pd.DataFrame([log_entry])
log.to_csv(log_path, index=False)

print(f"✅ Logged — {len(log)} total entries in paper_trading_log.csv")

# Updated portfolio
time.sleep(2)
acct2 = trading_client.get_account()
print(f"\nUpdated portfolio:")
print(f"  Portfolio: ${float(acct2.portfolio_value):,.2f}")
print(f"  Cash:      ${float(acct2.cash):,.2f}")

# Open positions
try:
    positions = trading_client.get_all_positions()
    if positions:
        print(f"\nOpen positions:")
        for p in positions:
            print(f"  {p.symbol}: {p.qty} shares  "
                  f"entry=${float(p.avg_entry_price):.2f}  "
                  f"P&L=${float(p.unrealized_pl):.2f}")
    else:
        print("\nNo open positions")
except Exception as e:
    print(f"Could not fetch positions: {e}")

✅ Logged — 3 total entries in paper_trading_log.csv

Updated portfolio:
  Portfolio: $94,564.47
  Cash:      $-99,411.18

Open positions:
  SPY: 263 shares  entry=$758.22  P&L=$-5435.52
